In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder 
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier 

In [48]:
df = pd.read_csv("Loan_Default_Cleaned.csv")
df.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [45]:
# Convert all object columns to category codes
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype('category').cat.codes

# Fill missing values
df = df.fillna(df.median(numeric_only=True))
df = df.fillna(df.mode().iloc[0])

# Ensure target column is integer
df["Status"] = df["Status"].astype(int)

In [41]:
cat_cols=df.select_dtypes(include=['object']).columns
cat_cols=cat_cols.drop('Status',errors='ignore')

In [47]:
df

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,0,3,0,0,0,0,0,1,...,3,758,0,0,1,98.728814,3,1,1,45.0
1,24891,2019,0,2,0,1,0,0,0,0,...,2,552,1,3,1,75.135870,0,1,1,39.0
2,24892,2019,0,2,1,0,0,0,0,1,...,3,834,0,1,1,80.019685,3,1,0,46.0
3,24893,2019,0,2,0,0,3,0,0,1,...,3,587,0,2,0,69.376900,0,1,0,42.0
4,24894,2019,0,1,1,0,0,0,0,1,...,1,602,1,0,0,91.886544,0,1,0,39.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148665,173555,2019,0,3,0,0,2,0,0,1,...,0,659,1,3,1,71.792763,3,1,0,48.0
148666,173556,2019,0,2,0,0,0,0,0,1,...,0,569,0,0,0,74.428934,3,1,0,15.0
148667,173557,2019,0,2,0,0,3,0,0,1,...,0,702,1,2,0,61.332418,0,1,0,49.0
148668,173558,2019,0,0,0,0,3,0,0,1,...,3,737,1,3,1,70.683453,0,1,0,29.0


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148670 entries, 0 to 148669
Data columns (total 34 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   ID                         148670 non-null  int64  
 1   year                       148670 non-null  int64  
 2   loan_limit                 148670 non-null  int8   
 3   Gender                     148670 non-null  int8   
 4   approv_in_adv              148670 non-null  int8   
 5   loan_type                  148670 non-null  int8   
 6   loan_purpose               148670 non-null  int8   
 7   Credit_Worthiness          148670 non-null  int8   
 8   open_credit                148670 non-null  int8   
 9   business_or_commercial     148670 non-null  int8   
 10  loan_amount                148670 non-null  int64  
 11  rate_of_interest           148670 non-null  float64
 12  Interest_rate_spread       148670 non-null  float64
 13  Upfront_charges            14

In [35]:
for col in cat_cols:
    print(col,df[col].unique())

In [5]:
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

In [33]:
y_col = 'Status'
x_cols = ['age','loan_amount','dependents','credit_history','previous_default','income_ratio']
x = df[x_cols]
y = df["Status"]

KeyError: "['dependents', 'credit_history', 'previous_default', 'income_ratio'] not in index"

In [21]:
x.columns

float(age),
            float(loan_amount),
            float(dependents),
            float(credit_history),
            float(previous_default),
            float(income_ratio)

['age','loan_amount','dependents','credit_history','previous_default','income_ratio']

Index(['loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose',
       'Credit_Worthiness', 'open_credit', 'business_or_commercial',
       'loan_amount', 'rate_of_interest', 'Interest_rate_spread',
       'Upfront_charges', 'term', 'Neg_ammortization', 'interest_only',
       'lump_sum_payment', 'property_value', 'construction_type',
       'occupancy_type', 'Secured_by', 'total_units', 'income', 'credit_type',
       'Credit_Score', 'co-applicant_credit_type', 'age',
       'submission_of_application', 'LTV', 'Region', 'Security_Type', 'dtir1'],
      dtype='object')

In [22]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [23]:
x_train

,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,rate_of_interest,...,income,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,dtir1
141039,0,3,0,0,3,0,0,1,206500,2.875,...,6360.0,3,676,1,4,1,27.242744,3,1,10.0
121276,0,0,0,0,0,0,0,1,466500,4.250,...,5100.0,0,675,0,2,0,69.835329,0,1,49.0
11214,0,0,0,0,2,0,0,1,326500,3.990,...,2760.0,0,593,0,3,0,46.115819,0,1,55.0
129659,0,1,0,0,3,0,0,1,256500,3.875,...,12300.0,3,894,1,3,1,89.062500,0,1,39.0
13370,0,2,0,0,0,0,0,1,166500,4.500,...,2100.0,0,896,0,4,1,76.376147,0,1,52.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98721,0,3,0,0,3,1,0,1,346500,3.750,...,6300.0,0,600,0,2,1,74.038462,3,1,44.0
44833,0,3,0,2,3,0,0,1,126500,3.500,...,5760.0,3,833,1,6,1,91.666667,3,1,39.0
113338,0,0,0,0,3,0,0,1,696500,3.625,...,11340.0,0,558,0,3,0,57.657285,0,1,23.0
39176,0,3,0,0,2,1,0,1,236500,3.990,...,1440.0,1,817,0,6,1,91.666667,3,1,14.0


In [24]:
from sklearn.linear_model import LogisticRegression

# Initialize Logistic Regression model
log_reg = LogisticRegression(
    random_state=42
)

# Train the model
log_reg.fit(x_train, y_train)

# Predict probability for positive class (class = 1)
y_pred_lr = log_reg.predict_proba(x_test)[:, 1]

# Calculate AUC-ROC score
auc_lr = roc_auc_score(y_test, y_pred_lr)

print("Logistic Regression AUC-ROC:", auc_lr)


Logistic Regression AUC-ROC: 0.5801252598434528


c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
xgb = XGBClassifier(eval_metric="logloss")

xgb.fit(x_train, y_train)
y_pred_xgb = xgb.predict_proba(x_test)[:,1]

auc_xgb = roc_auc_score(y_test, y_pred_xgb)
print("XGBoost AUC-ROC:", auc_xgb)

XGBoost AUC-ROC: 0.9999891407070081


In [9]:
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1]
}

grid_xgb = GridSearchCV(
    estimator=XGBClassifier(eval_metric="logloss"),
    param_grid=xgb_param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

grid_xgb.fit(x_train, y_train)

print("Best XGBoost Params:", grid_xgb.best_params_)
print("Best XGBoost AUC:", grid_xgb.best_score_)

Best XGBoost Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Best XGBoost AUC: 0.9999937109275434


In [30]:
lgbm = LGBMClassifier()

lgbm.fit(x_train, y_train)
y_pred_lgbm = lgbm.predict_proba(x_test)[:,1]

auc_lgbm = roc_auc_score(y_test, y_pred_lgbm)
print("LightGBM AUC-ROC:", auc_lgbm)

[LightGBM] [Info] Number of positive: 29311, number of negative: 89625
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007804 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1940
[LightGBM] [Info] Number of data points in the train set: 118936, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.246443 -> initscore=-1.117671
[LightGBM] [Info] Start training from score -1.117671
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

In [11]:
lgbm_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [-1, 5, 10],
    'learning_rate': [0.01, 0.1]
}

grid_lgbm = GridSearchCV(
    estimator=LGBMClassifier(),
    param_grid=lgbm_param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

grid_lgbm.fit(x_train, y_train)

print("Best LightGBM Params:", grid_lgbm.best_params_)
print("Best LightGBM AUC:", grid_lgbm.best_score_)

[LightGBM] [Info] Number of positive: 29311, number of negative: 89625
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012819 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2195
[LightGBM] [Info] Number of data points in the train set: 118936, number of used features: 32
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.246443 -> initscore=-1.117671
[LightGBM] [Info] Start training from score -1.117671
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

In [12]:
print("Baseline Logistic Regression AUC: <PUT YOUR VALUE HERE>")
print("XGBoost AUC:", auc_xgb)
print("LightGBM AUC:", auc_lgbm)

print("\nBest Tuned XGBoost AUC:", grid_xgb.best_score_)
print("Best Tuned LightGBM AUC:", grid_lgbm.best_score_)

Baseline Logistic Regression AUC: <PUT YOUR VALUE HERE>
XGBoost AUC: 0.9999907455436335
LightGBM AUC: 1.0

Best Tuned XGBoost AUC: 0.9999937109275434
Best Tuned LightGBM AUC: 0.9999939061456221


In [31]:
# save the model to disk
import joblib
filename = 'best_model.sav'
joblib.dump(lgbm, filename)

['best_model.sav']

In [32]:
# load the model from disk
loaded_model = joblib.load(filename)
result = loaded_model.score(x_test, y_test)
print(result)

1.0
